## HW2 (AutoChip) Example A: Binary-to-BCD (combinational)\n
\n
Run top-to-bottom. Only edit the API key cell (or set `OPENAI_API_KEY` in your environment).

In [16]:
import os

# IMPORTANT: Do not hardcode real API keys in this repository.
# Set `OPENAI_API_KEY` in your environment (e.g., export OPENAI_API_KEY="nvapi-REPLACE_ME").
# os.environ["OPENAI_API_KEY"] = "nvapi-REPLACE_ME"

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))


OPENAI_API_KEY set: True


In [17]:
# Colab/Ubuntu install helpers (optional)\n
# !apt-get update && apt-get install -y iverilog\n
# !pip -q install --upgrade openai\n

In [18]:
from pathlib import Path
import urllib.request
import json

ROOT = Path.cwd()
RUNS = ROOT / "runs" / "exampleA_binary_to_bcd"
RUNS.mkdir(parents=True, exist_ok=True)

tb_url = "https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v"
tb_path = RUNS / "binary_to_bcd_tb.v"
urllib.request.urlretrieve(tb_url, tb_path)

# --- Create AutoChip-style config.json + prompt.txt (for HW2 submission evidence) ---
prompt_path = RUNS / "prompt.txt"

prompt_text = """I am trying to create a Verilog model binary_to_bcd_converter for a binary to binary-coded-decimal converter. It must meet the following specifications:
	- Inputs:
		- Binary input (5-bits)
	- Outputs:
		- BCD (8-bits: 4-bits for the 10's place and 4-bits for the 1's place)

How would I write a design that meets these specifications?
"""

module_template = """\n// Module template (must match the testbench port names):\nmodule binary_to_bcd_converter (\n    input  [4:0] binary_input,\n    output [7:0] bcd_output\n);\n    // Insert code here\nendmodule\n"""

prompt_path.write_text(prompt_text + module_template, encoding="utf-8")

config_path = RUNS / "config.json"
config = {
    "general": {
        "prompt": "prompt.txt",
        "name": "binary_to_bcd_converter",
        "testbench": "binary_to_bcd_tb.v",
        "model_family": "NVIDIA",
        "model_id": "meta/llama-3.1-8b-instruct",
        "iterations": 5,
        "num_candidates": 4,
        "outdir": str(RUNS),
        "log": "run.log",
    }
}
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

(tb_path, prompt_path, config_path)


(PosixPath('/Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd/binary_to_bcd_tb.v'),
 PosixPath('/Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd/prompt.txt'),
 PosixPath('/Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd/config.json'))

In [19]:
# Load the original-ish prompt and module template from prompt.txt
prompt_path = RUNS / "prompt.txt"
DESIGN_PROMPT = prompt_path.read_text(encoding="utf-8")

print("Loaded DESIGN_PROMPT from:", prompt_path)


Loaded DESIGN_PROMPT from: /Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd/prompt.txt


In [20]:
from autochip_min import ensure_tooling, verilog_loop, DEFAULT_BASE_URL

import json
from pathlib import Path
import subprocess

ensure_tooling()

# Read AutoChip-style config.json for reproducibility
config = json.loads((RUNS / "config.json").read_text(encoding="utf-8"))
config_values = config["general"]

# Load prompt + testbench paths from config
prompt_path = RUNS / config_values["prompt"]
tb_path = RUNS / config_values["testbench"]

# Use NVIDIA integrate API
BASE_URL = DEFAULT_BASE_URL
MODEL_ID = config_values["model_id"]
MAX_ITERS = config_values["iterations"]
NUM_CANDIDATES = config_values["num_candidates"]

DESIGN_PROMPT = prompt_path.read_text(encoding="utf-8")

print("Loaded config.json from:", RUNS / "config.json")
print((RUNS / "config.json").read_text(encoding="utf-8"))

# --- AutoChip run (trajectory evidence) ---
best = None
try:
    best = verilog_loop(
        design_prompt=DESIGN_PROMPT,
        module_name=config_values["name"],
        tb_path=tb_path,
        outdir=RUNS,
        base_url=BASE_URL,
        model_id=MODEL_ID,
        max_iterations=MAX_ITERS,
        num_candidates=NUM_CANDIDATES,
    )

    print("Best iteration:", best.iteration, "candidate:", best.response_num)
    print("Compile OK:", best.sim.compile_ok, "Pass:", best.sim.ok)
    print("Command:\n" + best.sim.cmd)
    print("\nSTDOUT:\n", best.sim.stdout)
    print("\nSTDERR:\n", best.sim.stderr)
except Exception as e:
    # If AutoChip fails (missing key / server errors / non-convergence), we still
    # want the notebook to finish with a passing manual RTL check.
    print("AutoChip run failed:", repr(e))

# --- HW2 requirement: ensure passing RTL (manual fix) ---
# Use path relative to this homework folder so the notebook is robust.
manual_sv = RUNS.parent.parent / "binary_to_bcd_converter_manual.sv"
if not manual_sv.exists():
    raise FileNotFoundError(f"Missing manual RTL: {manual_sv}")

manual_out = RUNS / "manual_binary_to_bcd_sim.out"

compile_manual_cmd = [
    "iverilog",
    "-g2012",
    "-o",
    str(manual_out),
    str(tb_path),
    str(manual_sv),
]

# Evidence: show exact compilation/run commands for the report.
print("Manual iverilog command:\n" + " ".join(compile_manual_cmd))
print("Manual vvp command:\n" + "vvp " + str(manual_out))

proc = subprocess.run(compile_manual_cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print("Manual RTL compile stdout:\n", proc.stdout)
    print("Manual RTL compile stderr:\n", proc.stderr)
    raise RuntimeError("Manual RTL compile failed")

run_proc = subprocess.run(["vvp", str(manual_out)], capture_output=True, text=True)
print("\nManual RTL STDOUT:\n", run_proc.stdout)
print("\nManual RTL STDERR:\n", run_proc.stderr)

if "All test cases passed!" in run_proc.stdout:
    print("\nManual RTL: PASSED")
else:
    print("\nManual RTL: FAILED")


Loaded config.json from: /Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd/config.json
{
  "general": {
    "prompt": "prompt.txt",
    "name": "binary_to_bcd_converter",
    "testbench": "binary_to_bcd_tb.v",
    "model_family": "NVIDIA",
    "model_id": "meta/llama-3.1-8b-instruct",
    "iterations": 5,
    "num_candidates": 4,
    "outdir": "/Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd",
    "log": "run.log"
  }
}
Best iteration: 0 candidate: 0
Compile OK: True Pass: True
Command:
iverilog -g2012 -o /Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd/artifacts/sim_it0_c0/sim.out /Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd/binary_to_bcd_tb.v /Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/exampleA_binary_to_bcd/artifacts/binary_to_bcd_converter_it0_c0.sv && vvp /Users/princydoshi/Desktop/LLM4ChipDesign-1/HW2_AutoChip/runs/e